In [11]:
# Phase 4
# Using ML methods to compare with BL Regression

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np

In [7]:
train = pd.read_csv('../data/processed/race_driver_records_train_fe.csv')
test = pd.read_csv('../data/processed/race_driver_records_test_fe.csv')

selected_features = [
    'year', 'round', 'grid_position',
    'driver_podium_pct_y2', 'driver_avg_pos_y2', 'driver_podium_pct_y3',
    'constructor_races_y1', 'constructor_podium_pct_y2', 'constructor_podium_pct_y3',
    'driver_avg_pos_dry_3y', 'driver_avg_pos_wet_3y'
]

X_train = train[selected_features]
X_train = X_train.fillna(X_train.median())
y_train = train['is_podium']

X_test = test[selected_features]
X_test = X_test.fillna(X_test.median())
y_test = test['is_podium']

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"\nClass distribution (train): {y_train.value_counts().to_dict()}")
print(f"Class distribution (test): {y_test.value_counts().to_dict()}")

Train: 5857 samples, 11 features
Test: 240 samples, 11 features

Class distribution (train): {0: 5029, 1: 828}
Class distribution (test): {0: 204, 1: 36}


In [13]:
# Train Naive Bayes 

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_train_pred_proba = nb_model.predict_proba(X_train_scaled)[:, 1]
fpr_train, tpr_train, thresholds_train = roc_curve(y_train, y_train_pred_proba)
train_auc = roc_auc_score(y_train, y_train_pred_proba)

j_scores = tpr_train - fpr_train
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds_train[optimal_idx]


y_train_pred = (y_train_pred_proba >= optimal_threshold).astype(int)
cm_train = confusion_matrix(y_train, y_train_pred)

y_test_pred_proba = nb_model.predict_proba(X_test_scaled)[:, 1]
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_pred_proba)
test_auc = roc_auc_score(y_test, y_test_pred_proba)

y_test_pred = (y_test_pred_proba >= optimal_threshold).astype(int)
cm_test = confusion_matrix(y_test, y_test_pred)

print("="*60)
print("NAIVE BAYES RESULTS")
print("="*60)
print(f"\nOptimal Threshold: {optimal_threshold:.4f}")
print(f"\nTrain AUC: {train_auc:.4f}")
print(f"Train Confusion Matrix:\n{cm_train}")
print(f"\nTest AUC: {test_auc:.4f}")
print(f"Test Confusion Matrix:\n{cm_test}")

NAIVE BAYES RESULTS

Optimal Threshold: 0.0354

Train AUC: 0.8844
Train Confusion Matrix:
[[4022 1007]
 [ 130  698]]

Test AUC: 0.8952
Test Confusion Matrix:
[[151  53]
 [  4  32]]


In [15]:
# Method 2: Decision Tree
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)
dt_model.fit(X_train, y_train)

y_train_pred_proba_dt = dt_model.predict_proba(X_train)[:, 1]
fpr_train_dt, tpr_train_dt, thresholds_train_dt = roc_curve(y_train, y_train_pred_proba_dt)
train_auc_dt = roc_auc_score(y_train, y_train_pred_proba_dt)

j_scores_dt = tpr_train_dt - fpr_train_dt
optimal_idx_dt = np.argmax(j_scores_dt)
optimal_threshold_dt = thresholds_train_dt[optimal_idx_dt]

y_train_pred_dt = (y_train_pred_proba_dt >= optimal_threshold_dt).astype(int)
cm_train_dt = confusion_matrix(y_train, y_train_pred_dt)

y_test_pred_proba_dt = dt_model.predict_proba(X_test)[:, 1]
fpr_test_dt, tpr_test_dt, _ = roc_curve(y_test, y_test_pred_proba_dt)
test_auc_dt = roc_auc_score(y_test, y_test_pred_proba_dt)

y_test_pred_dt = (y_test_pred_proba_dt >= optimal_threshold_dt).astype(int)
cm_test_dt = confusion_matrix(y_test, y_test_pred_dt)

print("="*60)
print("DECISION TREE RESULTS")
print("="*60)
print(f"\nOptimal Threshold: {optimal_threshold_dt:.4f}")
print(f"\nTrain AUC: {train_auc_dt:.4f}")
print(f"Train Confusion Matrix:\n{cm_train_dt}")
print(f"\nTest AUC: {test_auc_dt:.4f}")
print(f"Test Confusion Matrix:\n{cm_test_dt}")

DECISION TREE RESULTS

Optimal Threshold: 0.1538

Train AUC: 0.9676
Train Confusion Matrix:
[[4350  679]
 [  33  795]]

Test AUC: 0.9459
Test Confusion Matrix:
[[160  44]
 [  4  32]]


In [17]:
# Method 3: Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_train_pred_proba_rf = rf_model.predict_proba(X_train)[:, 1]
fpr_train_rf, tpr_train_rf, thresholds_train_rf = roc_curve(y_train, y_train_pred_proba_rf)
train_auc_rf = roc_auc_score(y_train, y_train_pred_proba_rf)

j_scores_rf = tpr_train_rf - fpr_train_rf
optimal_idx_rf = np.argmax(j_scores_rf)
optimal_threshold_rf = thresholds_train_rf[optimal_idx_rf]

y_train_pred_rf = (y_train_pred_proba_rf >= optimal_threshold_rf).astype(int)
cm_train_rf = confusion_matrix(y_train, y_train_pred_rf)

y_test_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]
fpr_test_rf, tpr_test_rf, _ = roc_curve(y_test, y_test_pred_proba_rf)
test_auc_rf = roc_auc_score(y_test, y_test_pred_proba_rf)

y_test_pred_rf = (y_test_pred_proba_rf >= optimal_threshold_rf).astype(int)
cm_test_rf = confusion_matrix(y_test, y_test_pred_rf)

print("="*60)
print("RANDOM FOREST RESULTS")
print("="*60)
print(f"\nOptimal Threshold: {optimal_threshold_rf:.4f}")
print(f"\nTrain AUC: {train_auc_rf:.4f}")
print(f"Train Confusion Matrix:\n{cm_train_rf}")
print(f"\nTest AUC: {test_auc_rf:.4f}")
print(f"Test Confusion Matrix:\n{cm_test_rf}")

RANDOM FOREST RESULTS

Optimal Threshold: 0.1909

Train AUC: 0.9735
Train Confusion Matrix:
[[4458  571]
 [  36  792]]

Test AUC: 0.9314
Test Confusion Matrix:
[[164  40]
 [  4  32]]


In [19]:
# Method 4: SVM
svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    random_state=42
)
svm_model.fit(X_train_scaled, y_train)

y_train_pred_proba_svm = svm_model.predict_proba(X_train_scaled)[:, 1]
fpr_train_svm, tpr_train_svm, thresholds_train_svm = roc_curve(y_train, y_train_pred_proba_svm)
train_auc_svm = roc_auc_score(y_train, y_train_pred_proba_svm)

j_scores_svm = tpr_train_svm - fpr_train_svm
optimal_idx_svm = np.argmax(j_scores_svm)
optimal_threshold_svm = thresholds_train_svm[optimal_idx_svm]

y_train_pred_svm = (y_train_pred_proba_svm >= optimal_threshold_svm).astype(int)
cm_train_svm = confusion_matrix(y_train, y_train_pred_svm)

y_test_pred_proba_svm = svm_model.predict_proba(X_test_scaled)[:, 1]
fpr_test_svm, tpr_test_svm, _ = roc_curve(y_test, y_test_pred_proba_svm)
test_auc_svm = roc_auc_score(y_test, y_test_pred_proba_svm)

y_test_pred_svm = (y_test_pred_proba_svm >= optimal_threshold_svm).astype(int)
cm_test_svm = confusion_matrix(y_test, y_test_pred_svm)

print("="*60)
print("SVM (RBF) RESULTS")
print("="*60)
print(f"\nOptimal Threshold: {optimal_threshold_svm:.4f}")
print(f"\nTrain AUC: {train_auc_svm:.4f}")
print(f"Train Confusion Matrix:\n{cm_train_svm}")
print(f"\nTest AUC: {test_auc_svm:.4f}")
print(f"Test Confusion Matrix:\n{cm_test_svm}")

SVM (RBF) RESULTS

Optimal Threshold: 0.0814

Train AUC: 0.9177
Train Confusion Matrix:
[[4436  593]
 [  86  742]]

Test AUC: 0.8795
Test Confusion Matrix:
[[163  41]
 [  4  32]]


In [21]:
# Method 5: XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

y_train_pred_proba_xgb = xgb_model.predict_proba(X_train)[:, 1]
fpr_train_xgb, tpr_train_xgb, thresholds_train_xgb = roc_curve(y_train, y_train_pred_proba_xgb)
train_auc_xgb = roc_auc_score(y_train, y_train_pred_proba_xgb)

j_scores_xgb = tpr_train_xgb - fpr_train_xgb
optimal_idx_xgb = np.argmax(j_scores_xgb)
optimal_threshold_xgb = thresholds_train_xgb[optimal_idx_xgb]

y_train_pred_xgb = (y_train_pred_proba_xgb >= optimal_threshold_xgb).astype(int)
cm_train_xgb = confusion_matrix(y_train, y_train_pred_xgb)

y_test_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
fpr_test_xgb, tpr_test_xgb, _ = roc_curve(y_test, y_test_pred_proba_xgb)
test_auc_xgb = roc_auc_score(y_test, y_test_pred_proba_xgb)

y_test_pred_xgb = (y_test_pred_proba_xgb >= optimal_threshold_xgb).astype(int)
cm_test_xgb = confusion_matrix(y_test, y_test_pred_xgb)

print("="*60)
print("XGBOOST RESULTS")
print("="*60)
print(f"\nOptimal Threshold: {optimal_threshold_xgb:.4f}")
print(f"\nTrain AUC: {train_auc_xgb:.4f}")
print(f"Train Confusion Matrix:\n{cm_train_xgb}")
print(f"\nTest AUC: {test_auc_xgb:.4f}")
print(f"Test Confusion Matrix:\n{cm_test_xgb}")

XGBOOST RESULTS

Optimal Threshold: 0.2780

Train AUC: 0.9928
Train Confusion Matrix:
[[4758  271]
 [  20  808]]

Test AUC: 0.9206
Test Confusion Matrix:
[[189  15]
 [ 11  25]]


In [23]:
# Create comparison dataframe
comparison_data = {
    'Model': ['Naïve Bayes', 'Decision Tree', 'Random Forest', 'SVM', 'XGBoost'],
    'Train AUC': [train_auc, train_auc_dt, train_auc_rf, train_auc_svm, train_auc_xgb],
    'Test AUC': [test_auc, test_auc_dt, test_auc_rf, test_auc_svm, test_auc_xgb],
    'Optimal Threshold': [optimal_threshold, optimal_threshold_dt, optimal_threshold_rf,
                          optimal_threshold_svm, optimal_threshold_xgb]
}
comparison_df = pd.DataFrame(comparison_data)

comparison_df['Overfitting (∆AUC)'] = comparison_df['Train AUC'] - comparison_df['Test AUC']

comparison_df = comparison_df.sort_values('Test AUC', ascending=False)

print("="*80)
print("MODEL COMPARISON - Sorted by Test AUC")
print("="*80)
print(comparison_df.to_string(index=False))
print("\n")

best_model = comparison_df.iloc[0]
print("="*80)
print(f"BEST MODEL: {best_model['Model']}")
print(f"Test AUC: {best_model['Test AUC']:.4f}")
print(f"Train AUC: {best_model['Train AUC']:.4f}")
print(f"Overfitting: {best_model['Overfitting (∆AUC)']:.4f}")
print("="*80)


MODEL COMPARISON - Sorted by Test AUC
        Model  Train AUC  Test AUC  Optimal Threshold  Overfitting (∆AUC)
Decision Tree   0.967629  0.945942           0.153846            0.021686
Random Forest   0.973534  0.931373           0.190948            0.042161
      XGBoost   0.992847  0.920615           0.278002            0.072231
  Naïve Bayes   0.884395  0.895153           0.035402           -0.010757
          SVM   0.917696  0.879493           0.081368            0.038203


BEST MODEL: Decision Tree
Test AUC: 0.9459
Train AUC: 0.9676
Overfitting: 0.0217


**Discussion:**
Decision tree appears to be the best model for predicting top 3 finishers. Decision tree works by examining the dataset and checking for the feature which best separates a podium finisher from a non-podium finisher. It then repeats this recursively, until it reaches it's desired depth (max_depth=10 here). 


It is interesting that random forest performed worse than decision tree, as random forest is essentially a collection of decision trees. Perhaps this is due to the fact that the data set is relatively small, or that F1 has such a powerful predictor in "grid position" that a less complex prediction method will suffice. 